<a href="https://colab.research.google.com/github/BriennedeTarth/opensees_demo/blob/main/opensees.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install openseespy
import openseespy.opensees as ops
import math # Added for PI
# Define units based on the provided Tcl script and standard conversions
# Basic units
m   = 1.0   # meter for length
sec = 1.0   # second for time
kg  = 1.0   # Kilogram for mass

# Angle units
rad = 1.0
deg = (math.pi / 180.0) * rad

# Length units
cm  = 0.01
mm  = 0.001 # Derived: 1/1000 of a meter
inch = 0.0254 # Using 'inch' to avoid conflict with Python keyword if any, and for clarity
ft = 12.0 * inch

# Mass units
lbs = 0.4536
kip = 1000.0 * lbs
ton = 1000.0 * kg

# Force units
N   = kg * m / (sec * sec)
kN = 1000.0 * N
# Derived force units (standard gravity 9.80665 m/s^2)
g_accel = 9.80665 # standard gravity acceleration in m/s^2
kgf = kg * g_accel # kilogram-force
tf = 1000.0 * kgf # tonne-force

# Pressure units
Pa  = 1.0 * N / (m**2)
kPa = 1000.0 * Pa
MPa = 1e6 * Pa # Derived: MegaPascal
pcf = lbs / (ft**3)	# pcf = #/cubic foot
ksi = kip / (inch**2)
psi = ksi / 1000.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.3/86.3 MB 11.3 MB/s eta 0:00:00


In [ ]:
import math # Added for PI

L=6*m
H=3*m
# ssection
b = 0.3 * m
h = 0.3 * m

A = b * h
I = b * h**3 / 12

fc=25*MPa
E=4700*(fc)**0.5
P=10*tf




In [ ]:
# Define the model
# The model is a 2D concrete frame, to be modeled using elestic linear alalysis
# The columns are 3m tall and the bay is 6m in length
# Al elements are rectangular concrete sections of size 30x30
# The load is aapliad as a later load on the left node
# The load will be defined as a linear ramp from 0 to 1
# Using a consisten unit system in N-mm


# model('basic', '-ndm', ndm, '-ndf', ndf=ndm*(ndm+1)/2
ops.model('basic', '-ndm', 2, '-ndf', 3)
ops.wipe()

# Define time series
# timeSeries('Linear', tag, '-factor', factor=1.0)
linear_time_series = 1
ops.timeSeries('Linear', linear_time_series, '-factor', 1.0)

# Define the Geometry
# node(nodeTag, *crds, '-ndf', ndf, '-mass', *mass, '-disp', *disp, '-vel', *vel, '-accel', *accel)¶
node_1 = 1
coords=[0, 0]
ops.node(node_1, *coords)

node_2 = 2
coords=[0, H]
ops.node(node_2, *coords)

node_3 = 3
coords=[L, H]
ops.node(node_3, *coords)

node_4 = 4
coords=[L, 0]
ops.node(node_4, *coords)

# Element definition
# element('elasticBeamColumn', eleTag, *eleNodes, secTag, transfTag, <'-mass', mass>, <'-cMass'>, <'-release', releaseCode>)

# section('Elastic', secTag, E_mod, A, Iz, G_mod=None, alphaY=None)

# geomTransf(transfType, transfTag, *transfArgs)

section_1 = 1
ops.section('Elastic', section_1, E, A, I)

geoTransformation = 1
ops.geomTransf('Linear', geoTransformation)

col_1 = 1
ops.element('elasticBeamColumn', col_1, node_1, node_2, section_1, geoTransformation)
col_2 = 2
ops.element('elasticBeamColumn', col_2, node_4, node_3, section_1, geoTransformation)
beam_1 = 3
ops.element('elasticBeamColumn', beam_1, node_2, node_3, section_1, geoTransformation)

# Define the boundary conditions
# fix(nodeTag, *constrValues)
ops.fix(node_1, *[1,1,1])
ops.fix(node_4, *[1,1,0])

# Define the load pattern
# pattern('Plain', patternTag, tsTag, '-fact', fact)
load_pattern = 1
ops.pattern('Plain', load_pattern, linear_time_series, '-fact', 1.0)
# load(nodeTag, *loadValues)
ops.load(node_2, *[P, 0, 0])

# Define the analysis
# constraints(constraintType, *constraintArgs)
ops.constraints('Transformation')
# numberer(numbererType, *numbererArgs)
ops.numberer('RCM')
# system(systemType, *systemArgs)
ops.system('UmfPack')
# test(testType, *testArgs)
ops.test('NormUnbalance', 1e-8, 10)
# algorithm('Linear', secant=False, initial=False, factorOnce=False)
ops.algorithm('Linear', 'factorOnce', True)
# integrator('LoadControl', incr, numIter=1, minIncr=incr, maxIncr=incr)
ops.integrator('LoadControl', 1)
# analysis(analysisType)
ops.analysis('Static')

ops.analyze(1)

0

In [ ]:
ops.nodeDisp(node_2)
ops.nodeDisp(node_3)

[20.39505794174958, -0.03430077487151462, -2.814904558575712]